# 01 — Data Pipeline

Questo notebook costruisce il dataset di coppie `(clean_mel, degraded_mel)` utilizzato per addestrare la U-Net di restauro audio.

**Flusso generale:**
1. Scarica FMA-small (dataset audio libero, 8000 tracce ~30s)
2. Carica EnCodec 24 kHz per degradare l'audio (simula gli artefatti RVQ che MusicGen produce)
3. Per ogni traccia: estrai segmento → encode→decode con EnCodec → calcola Mel spectrogrammi clean e degraded
4. Salva le coppie su Google Drive come tensori `.pt`
5. Verifica integrità e visualizza campioni

## Cella 1 — Installazione dipendenze

- `encodec`: modello Meta per compressione/degradazione audio neurale
- `torchaudio`: I/O audio e trasformazioni (MelSpectrogram)
- `tqdm`: progress bar per il loop principale
- `huggingface_hub`: fallback per scaricare FMA se il server ufficiale non risponde

In [1]:
import subprocess, sys

def pip_install(*packages):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])

pip_install("encodec", "torchaudio", "tqdm", "huggingface_hub")
print("Dipendenze installate.")

Dipendenze installate.


## Cella 2 — Mount Google Drive e creazione cartelle

I file `.pt` delle coppie vengono salvati su Drive per persistere tra le sessioni Colab.
`DATA_DIR` viene creata se non esiste già (idempotente).

In [2]:
import os
from google.colab import drive

DRIVE_ROOT      = "/content/drive/MyDrive/audio-restoration"
DATA_DIR        = f"{DRIVE_ROOT}/data/train"

drive.mount("/content/drive", force_remount=False)

os.makedirs(DATA_DIR, exist_ok=True)
print(f"Drive montato. Cartella output: {DATA_DIR}")
print(f"File già presenti: {len([f for f in os.listdir(DATA_DIR) if f.endswith('.pt')])}")

Mounted at /content/drive
Drive montato. Cartella output: /content/drive/MyDrive/audio-restoration/data/train
File già presenti: 2000


## Cella 3 — Download FMA-small

FMA-small contiene 8000 tracce MP3 a 128 kbps (~30s) suddivise in 8 generi.

**Strategia:**
1. Tentativo primario: `wget` dal server ufficiale `os.unil.ch` (~7.2 GB)
2. Fallback: `huggingface_hub.snapshot_download` da un mirror HuggingFace

Il file zip viene decompresso in `FMA_LOCAL`; se la cartella esiste già il download viene saltato.

In [3]:
# Cella 3 — Download FMA-small via Kaggle API
# Usiamo Colab Secrets per la key (sicuro, non esposta nel codice)

from google.colab import userdata
import os
import subprocess

# Leggi token da Colab Secrets
os.environ['KAGGLE_TOKEN'] = userdata.get('KAGGLE_TOKEN')

# Installa kaggle
!pip install -q kaggle

# Scarica FMA-small
!kaggle datasets download \
  -d imsparsh/fma-free-music-archive-small-medium \
  -p /content/

# Decomprimi
!unzip -q /content/fma-free-music-archive-small-medium.zip \
  -d /content/fma_small/

# Conta MP3
import glob
mp3_files = glob.glob('/content/fma_small/**/*.mp3', recursive=True)
n_mp3 = len(mp3_files)
print(f"\nMP3 trovati: {n_mp3}")
assert n_mp3 > 0, "Nessun MP3 trovato — verifica il download"

Dataset URL: https://www.kaggle.com/datasets/imsparsh/fma-free-music-archive-small-medium
License(s): CC0-1.0
100% 29.8G/29.8G [03:19<00:00, 161MB/s]


MP3 trovati: 33000


## Cella 4 — Caricamento modello EnCodec

EnCodec 24 kHz è il codec neurale di Meta usato internamente da MusicGen.
Con `bandwidth=6.0` kbps vengono attivati 8 codebook RVQ → il segnale decodificato
presenta gli stessi artefatti che vediamo nell'audio generato da MusicGen.

Il modello viene messo in `eval()` e i gradienti disabilitati: serve solo per l'inferenza.

In [4]:
import torch
from encodec import EncodecModel

BANDWIDTH = 6.0

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

encodec_model = EncodecModel.encodec_model_24khz()
encodec_model.set_target_bandwidth(BANDWIDTH)
encodec_model = encodec_model.to(device).eval()

for param in encodec_model.parameters():
    param.requires_grad_(False)

print(f"EnCodec caricato su: {device}")
print(f"Bandwidth impostata: {BANDWIDTH} kbps")
if device.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name}  ({props.total_memory // 1024**2} MB VRAM)")

/usr/local/lib/python3.12/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Downloading: "https://dl.fbaipublicfiles.com/encodec/v0/encodec_24khz-d7cc33bc.th" to /root/.cache/torch/hub/checkpoints/encodec_24khz-d7cc33bc.th


100%|██████████| 88.9M/88.9M [00:00<00:00, 214MB/s]


EnCodec caricato su: cuda
Bandwidth impostata: 6.0 kbps
GPU: NVIDIA L4  (22563 MB VRAM)


## Cella A — Configurazione v2 e process_track_v2

Nuovi parametri allineati al vocoder **Vocos** (`vocos-mel-24khz`):
- `n_mels=100` (Vocos richiede esattamente 100 bande Mel)
- 5 bandwidth per **multi-bandwidth degradation**: moltiplica il dataset senza scaricare altro audio
- Salvato in `train_v2/` — `train/` esistente **non viene toccato**

> `process_track_v2` è indipendente da `process_track`. Non sovrascrive nulla.

In [5]:
import os
import torch
import torchaudio
import torchaudio.transforms as T
from typing import Optional, Dict

# ── Percorsi v2 ───────────────────────────────────────────────────────────────
DRIVE_ROOT_V2       = "/content/drive/MyDrive/audio-restoration"
DATA_DIR_V2         = f"{DRIVE_ROOT_V2}/data/train_v2"

# ── Audio (invariati) ─────────────────────────────────────────────────────────
V2_TARGET_SR        = 24_000
V2_SEGMENT_SAMPLES  = 96_000

# ── Mel v2 — compatibile con vocos-mel-24khz ──────────────────────────────────
V2_N_MELS           = 100
V2_N_FFT            = 1024
V2_HOP_LENGTH       = 256
V2_F_MIN            = 0
V2_F_MAX            = 12_000

# ── Multi-bandwidth ───────────────────────────────────────────────────────────
BANDWIDTHS          = [1.5, 3.0, 6.0, 12.0, 24.0]
TARGET_PAIRS_PER_BW = 2000

os.makedirs(DATA_DIR_V2, exist_ok=True)
print(f"Dataset v2 directory : {DATA_DIR_V2}")
print(f"N_MELS v2            : {V2_N_MELS}")
print(f"Bandwidths           : {BANDWIDTHS}")
print(f"Coppie per bandwidth : {TARGET_PAIRS_PER_BW}")
print(f"Totale coppie attese : {len(BANDWIDTHS) * TARGET_PAIRS_PER_BW}")


def _build_mel_v2(device: torch.device) -> T.MelSpectrogram:
    return T.MelSpectrogram(
        sample_rate=V2_TARGET_SR,
        n_fft=V2_N_FFT,
        hop_length=V2_HOP_LENGTH,
        n_mels=V2_N_MELS,
        f_min=V2_F_MIN,
        f_max=V2_F_MAX,
        power=1.0,
        center=True,
        pad_mode="reflect",
        norm="slaney",
        mel_scale="slaney",
    ).to(device)


# Istanziata UNA SOLA VOLTA — riusata per tutte le 10.000 tracce del loop.
# Evita overhead di init e allocazioni GPU ripetute.
mel_fn_v2 = _build_mel_v2(device)


def process_track_v2(
    mp3_path: str,
    model,
    device: torch.device,
    mel_fn: T.MelSpectrogram,
) -> Optional[Dict[str, torch.Tensor]]:
    """
    Processa una traccia con i parametri v2 (n_mels=100).
    Il modello EnCodec deve avere la bandwidth gia impostata esternamente
    tramite codec.set_target_bandwidth(bw) prima di chiamare questa funzione.
    mel_fn: trasformata MelSpectrogram pre-istanziata (condivisa tra le chiamate).

    Returns:
        dict {clean_mel, degraded_mel} shape [1, 100, 376] oppure None.
    """
    try:
        waveform, sr = torchaudio.load(mp3_path)

        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)

        if sr != V2_TARGET_SR:
            resampler = T.Resample(orig_freq=sr, new_freq=V2_TARGET_SR)
            waveform  = resampler(waveform)

        n_samples = waveform.shape[-1]
        if n_samples < V2_SEGMENT_SAMPLES:
            return None
        start = (
            0 if n_samples == V2_SEGMENT_SAMPLES
            else torch.randint(0, n_samples - V2_SEGMENT_SAMPLES, (1,)).item()
        )
        waveform = waveform[:, start : start + V2_SEGMENT_SAMPLES]

        wav_enc = waveform.unsqueeze(0).to(device)
        with torch.no_grad():
            encoded_frames = model.encode(wav_enc)
            degraded_wav   = model.decode(encoded_frames)

        clean_wav    = waveform.to(device)
        degraded_wav = degraded_wav.squeeze(0)

        min_len      = min(clean_wav.shape[-1], degraded_wav.shape[-1])
        clean_wav    = clean_wav[..., :min_len]
        degraded_wav = degraded_wav[..., :min_len]

        log_mel = lambda x: torch.log(torch.clamp(mel_fn(x), min=1e-5))

        return {
            "clean_mel":    log_mel(clean_wav).cpu(),
            "degraded_mel": log_mel(degraded_wav).cpu(),
        }

    except Exception:
        return None


print("process_track_v2 definita.")
print(f"  Firma: process_track_v2(mp3_path, model, device, mel_fn)")
print(f"  Shape attesa per tensore: [1, {V2_N_MELS}, 376]")


Dataset v2 directory : /content/drive/MyDrive/audio-restoration/data/train_v2
N_MELS v2            : 100
Bandwidths           : [1.5, 3.0, 6.0, 12.0, 24.0]
Coppie per bandwidth : 2000
Totale coppie attese : 10000
process_track_v2 definita.
  Firma: process_track_v2(mp3_path, model, device, mel_fn)
  Shape attesa per tensore: [1, 100, 376]


## Cella B — Loop multi-bandwidth

Per ciascuna delle 5 bandwidth processa `TARGET_PAIRS_PER_BW` coppie e le salva in `train_v2/`.

**Strategia EnCodec**: il modello rimane in GPU — si chiama solo
`encodec_model.set_target_bandwidth(bw)` tra un loop e l'altro.
Evita il reload del modello (~89 MB) prevenendo memory leak e OOM su T4.

Naming `pair_bw6.0_0042.pt` identifica univocamente bandwidth e indice.

In [6]:
import os
import glob
import time
import random
import torch
from tqdm.auto import tqdm

# Usa la lista MP3 gia in memoria. Se assente, riscansiona.
FMA_LOCAL = "/content/fma_small"
if "all_mp3s" not in globals() or len(all_mp3s) == 0:
    all_mp3s = glob.glob(os.path.join(FMA_LOCAL, "**", "*.mp3"), recursive=True)
    print(f"MP3 trovati: {len(all_mp3s)}")

CHECKPOINT_EVERY = 200

for bw in BANDWIDTHS:
    print(f"\n{'='*60}")
    print(f"Bandwidth: {bw} kbps")
    print(f"{'='*60}")

    # Cambia bandwidth sul modello gia in GPU — nessun reload
    encodec_model.set_target_bandwidth(bw)

    tag     = f"bw{bw}"
    pattern = os.path.join(DATA_DIR_V2, f"pair_{tag}_*.pt")
    already = len(glob.glob(pattern))
    print(f"Gia salvate: {already} / {TARGET_PAIRS_PER_BW}")

    if already >= TARGET_PAIRS_PER_BW:
        print("  Completo. Skip.")
        continue

    rng_seed = int(bw * 10)
    mp3_list = all_mp3s[:]
    random.seed(rng_seed)
    random.shuffle(mp3_list)

    saved   = already
    skipped = 0
    t_start = time.time()

    pbar = tqdm(total=TARGET_PAIRS_PER_BW - already, desc=f"bw={bw}", unit="pair")

    for mp3_path in mp3_list:
        if saved >= TARGET_PAIRS_PER_BW:
            break

        try:
            result = process_track_v2(mp3_path, encodec_model, device, mel_fn_v2)
        except Exception:
            result = None

        if result is None:
            skipped += 1
            continue

        fname     = f"pair_{tag}_{saved:04d}.pt"
        save_path = os.path.join(DATA_DIR_V2, fname)
        torch.save(result, save_path)
        saved += 1
        pbar.update(1)

        if saved % CHECKPOINT_EVERY == 0:
            elapsed   = time.time() - t_start
            done      = saved - already
            rate      = done / elapsed if elapsed > 0 else 1
            remaining = (TARGET_PAIRS_PER_BW - saved) / rate
            tqdm.write(
                f"  [ckpt bw={bw}] saved={saved:4d}  skip={skipped}  "
                f"elapsed={elapsed/60:.1f}min  ~rem={remaining/60:.1f}min"
            )

    pbar.close()
    elapsed_tot = time.time() - t_start
    print(f"  Salvate: {saved}  |  Skip: {skipped}  |  Tempo: {elapsed_tot/60:.1f} min")

print("\nLoop multi-bandwidth completato.")


MP3 trovati: 33000

Bandwidth: 1.5 kbps
Gia salvate: 0 / 2000


bw=1.5:   0%|          | 0/2000 [00:00<?, ?pair/s]

  [ckpt bw=1.5] saved= 200  skip=0  elapsed=0.5min  ~rem=4.1min
  [ckpt bw=1.5] saved= 400  skip=1  elapsed=0.8min  ~rem=3.1min
  [ckpt bw=1.5] saved= 600  skip=1  elapsed=1.1min  ~rem=2.6min
  [ckpt bw=1.5] saved= 800  skip=2  elapsed=1.4min  ~rem=2.2min
  [ckpt bw=1.5] saved=1000  skip=2  elapsed=1.8min  ~rem=1.8min
  [ckpt bw=1.5] saved=1200  skip=2  elapsed=2.1min  ~rem=1.4min
  [ckpt bw=1.5] saved=1400  skip=3  elapsed=2.4min  ~rem=1.0min
  [ckpt bw=1.5] saved=1600  skip=3  elapsed=2.7min  ~rem=0.7min
  [ckpt bw=1.5] saved=1800  skip=3  elapsed=3.1min  ~rem=0.3min
  [ckpt bw=1.5] saved=2000  skip=3  elapsed=3.4min  ~rem=0.0min
  Salvate: 2000  |  Skip: 3  |  Tempo: 3.4 min

Bandwidth: 3.0 kbps
Gia salvate: 0 / 2000


bw=3.0:   0%|          | 0/2000 [00:00<?, ?pair/s]

  [ckpt bw=3.0] saved= 200  skip=0  elapsed=0.3min  ~rem=3.0min
  [ckpt bw=3.0] saved= 400  skip=0  elapsed=0.7min  ~rem=2.6min
  [ckpt bw=3.0] saved= 600  skip=0  elapsed=1.0min  ~rem=2.3min
  [ckpt bw=3.0] saved= 800  skip=0  elapsed=1.3min  ~rem=2.0min
  [ckpt bw=3.0] saved=1000  skip=1  elapsed=1.7min  ~rem=1.7min
  [ckpt bw=3.0] saved=1200  skip=1  elapsed=2.0min  ~rem=1.3min
  [ckpt bw=3.0] saved=1400  skip=1  elapsed=2.3min  ~rem=1.0min
  [ckpt bw=3.0] saved=1600  skip=1  elapsed=2.6min  ~rem=0.7min
  [ckpt bw=3.0] saved=1800  skip=1  elapsed=3.0min  ~rem=0.3min
  [ckpt bw=3.0] saved=2000  skip=1  elapsed=3.3min  ~rem=0.0min
  Salvate: 2000  |  Skip: 1  |  Tempo: 3.3 min

Bandwidth: 6.0 kbps
Gia salvate: 0 / 2000


...

  [ckpt bw=6.0] saved= 200  skip=0  elapsed=0.3min  ~rem=3.0min
  [ckpt bw=6.0] saved= 400  skip=0  elapsed=0.7min  ~rem=2.7min
  [ckpt bw=6.0] saved= 600  skip=1  elapsed=1.0min  ~rem=2.4min
  [ckpt bw=6.0] saved= 800  skip=1  elapsed=1.3min  ~rem=2.0min
  [ckpt bw=6.0] saved=1000  skip=3  elapsed=1.7min  ~rem=1.7min
  [ckpt bw=6.0] saved=1200  skip=3  elapsed=2.0min  ~rem=1.3min
  [ckpt bw=6.0] saved=1400  skip=3  elapsed=2.3min  ~rem=1.0min
  [ckpt bw=6.0] saved=1600  skip=3  elapsed=2.7min  ~rem=0.7min
  [ckpt bw=6.0] saved=1800  skip=3  elapsed=3.0min  ~rem=0.3min
  [ckpt bw=6.0] saved=2000  skip=3  elapsed=3.3min  ~rem=0.0min
  Salvate: 2000  |  Skip: 3  |  Tempo: 3.3 min

Bandwidth: 12.0 kbps
Gia salvate: 0 / 2000


bw=12.0:   0%|          | 0/2000 [00:00<?, ?pair/s]

  [ckpt bw=12.0] saved= 200  skip=0  elapsed=0.3min  ~rem=3.1min
  [ckpt bw=12.0] saved= 400  skip=0  elapsed=0.7min  ~rem=2.8min
  [ckpt bw=12.0] saved= 600  skip=0  elapsed=1.0min  ~rem=2.4min
  [ckpt bw=12.0] saved= 800  skip=0  elapsed=1.4min  ~rem=2.1min
  [ckpt bw=12.0] saved=1000  skip=0  elapsed=1.7min  ~rem=1.7min
  [ckpt bw=12.0] saved=1200  skip=0  elapsed=2.1min  ~rem=1.4min
  [ckpt bw=12.0] saved=1400  skip=1  elapsed=2.4min  ~rem=1.0min
  [ckpt bw=12.0] saved=1600  skip=1  elapsed=2.8min  ~rem=0.7min
  [ckpt bw=12.0] saved=1800  skip=2  elapsed=3.1min  ~rem=0.3min
  [ckpt bw=12.0] saved=2000  skip=2  elapsed=3.5min  ~rem=0.0min
  Salvate: 2000  |  Skip: 2  |  Tempo: 3.5 min

Bandwidth: 24.0 kbps
Gia salvate: 0 / 2000


bw=24.0:   0%|          | 0/2000 [00:00<?, ?pair/s]

  [ckpt bw=24.0] saved= 200  skip=0  elapsed=0.4min  ~rem=3.3min
  [ckpt bw=24.0] saved= 400  skip=0  elapsed=0.7min  ~rem=3.0min
  [ckpt bw=24.0] saved= 600  skip=0  elapsed=1.1min  ~rem=2.6min
  [ckpt bw=24.0] saved= 800  skip=0  elapsed=1.5min  ~rem=2.2min
  [ckpt bw=24.0] saved=1000  skip=0  elapsed=1.9min  ~rem=1.9min
  [ckpt bw=24.0] saved=1200  skip=0  elapsed=2.2min  ~rem=1.5min
  [ckpt bw=24.0] saved=1400  skip=1  elapsed=2.6min  ~rem=1.1min
  [ckpt bw=24.0] saved=1600  skip=1  elapsed=2.9min  ~rem=0.7min
  [ckpt bw=24.0] saved=1800  skip=1  elapsed=3.3min  ~rem=0.4min
  [ckpt bw=24.0] saved=2000  skip=1  elapsed=3.7min  ~rem=0.0min
  Salvate: 2000  |  Skip: 1  |  Tempo: 3.7 min

Loop multi-bandwidth completato.


## Cella C — Verifica finale dataset v2

Per ogni bandwidth:
- Conta le coppie salvate
- Verifica shape `[1, 100, 376]` su un campione casuale
- Calcola min / max / mean / LSD

Il LSD deve crescere al diminuire della bandwidth:
bw=1.5 (massima compressione) → LSD alto,
bw=24.0 (minima compressione) → LSD vicino a 0.

In [7]:
import os
import glob
import random
import torch

DRIVE_ROOT_V2  = "/content/drive/MyDrive/audio-restoration"
DATA_DIR_V2    = f"{DRIVE_ROOT_V2}/data/train_v2"
BANDWIDTHS     = [1.5, 3.0, 6.0, 12.0, 24.0]
EXPECTED_SHAPE = (1, 100, 376)


def lsd(clean: torch.Tensor, degraded: torch.Tensor) -> float:
    return (clean - degraded).pow(2).mean(dim=-1).sqrt().mean().item()


header = (
    f"{'Bandwidth':>12} {'Coppie':>8} {'Shape':>10} "
    f"{'Min':>8} {'Max':>8} {'Mean':>8} {'LSD':>8}"
)
print(f"Verifica dataset v2 : {DATA_DIR_V2}")
print(f"Shape attesa        : {EXPECTED_SHAPE}")
print()
print(header)
print("-" * len(header))

total_pairs = 0
all_ok      = True

for bw in BANDWIDTHS:
    tag   = f"bw{bw}"
    pairs = sorted(glob.glob(os.path.join(DATA_DIR_V2, f"pair_{tag}_*.pt")))
    n     = len(pairs)
    total_pairs += n

    if n == 0:
        print(f"{bw:>10} kbps {n:>8} {'N/A':>10}  {'---':>7} {'---':>8} {'---':>8} {'---':>8}")
        all_ok = False
        continue

    data      = torch.load(random.choice(pairs), weights_only=True)
    c_mel     = data["clean_mel"]
    d_mel     = data["degraded_mel"]
    shape_ok  = tuple(c_mel.shape) == EXPECTED_SHAPE and tuple(d_mel.shape) == EXPECTED_SHAPE
    if not shape_ok:
        all_ok = False
    shape_str = "OK" if shape_ok else f"FAIL{tuple(c_mel.shape)}"
    lsd_val   = lsd(c_mel, d_mel)

    print(
        f"{bw:>10} kbps {n:>8} {shape_str:>10} "
        f"{c_mel.min().item():>8.3f} {c_mel.max().item():>8.3f} "
        f"{c_mel.mean().item():>8.3f} {lsd_val:>8.4f}"
    )

print("-" * len(header))
print(f"{'TOTALE':>12}  {total_pairs:>8}")
print()
if all_ok:
    print("Tutti i controlli superati. Dataset v2 pronto per il training.")
else:
    print("ATTENZIONE: alcuni controlli falliti.")


Verifica dataset v2 : /content/drive/MyDrive/audio-restoration/data/train_v2
Shape attesa        : (1, 100, 376)

   Bandwidth   Coppie      Shape      Min      Max     Mean      LSD
--------------------------------------------------------------------
       1.5 kbps     2000         OK   -7.156    1.705   -3.174   0.5718
       3.0 kbps     2000         OK   -8.792    1.254   -2.559   0.4892
       6.0 kbps     2000         OK   -7.382    1.464   -3.121   0.6751
      12.0 kbps     2000         OK   -9.307    2.158   -2.873   0.5043
      24.0 kbps     2000         OK   -8.672    0.922   -3.005   0.3223
--------------------------------------------------------------------
      TOTALE     10000

Tutti i controlli superati. Dataset v2 pronto per il training.
